# LoRA Fine-tuning of Large Language Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/surrey-nlp/NLP-2026/blob/main/lab09/lab09_LoRA_FT_with_LLMs.ipynb)

## Overview

This notebook demonstrates **Low-Rank Adaptation (LoRA)** for efficient fine-tuning of large language models on a **sentiment analysis** task. LoRA is a technique that significantly reduces the number of trainable parameters while maintaining competitive performance.

**Task:** Binary sentiment classification (positive / negative) on the [IMDB Movie Reviews dataset](https://huggingface.co/datasets/imdb).

**What you'll learn:**
- Load and explore the IMDB sentiment analysis dataset
- Establish a zero-shot baseline with an untuned LLM
- Configure and apply LoRA for parameter-efficient fine-tuning
- Evaluate and compare sentiment classification accuracy before and after fine-tuning

**Key concept:** Instead of fine-tuning all parameters of a large model, LoRA freezes the original weights and injects trainable low-rank matrices into selected layers. This dramatically reduces memory usage and computational cost while achieving strong results.

**Sources:**
- [Hugging Face PEFT Blog](https://huggingface.co/blog/peft)
- [PEFT LoRA Developer Guide](https://huggingface.co/docs/peft/developer_guides/lora)


## Setup: Install Required Libraries

In [ ]:
# Install required packages
# Note: This may take a couple of minutes
import subprocess
import sys

packages = [
    "transformers>=4.51.0",
    "peft>=0.11.0",
    "datasets>=2.10.0",
    "torch>=2.1.0",
    "accelerate>=0.30.0",
    "tqdm"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("All packages installed successfully!")

## Import Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
import numpy as np
from tqdm import tqdm

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load the Dataset

We'll use the IMDB dataset, a benchmark for binary sentiment classification (positive/negative reviews).

In [ ]:
# Load IMDB dataset
dataset = load_dataset("imdb")

# Display dataset info
print(f"Train set size: {len(dataset['train'])}")
print(f"Test set size: {len(dataset['test'])}")
print(f"\nExample (first training sample):")
print(f"Text: {dataset['train'][0]['text'][:200]}...")
print(f"Label: {dataset['train'][0]['label']} (0=negative, 1=positive)")

## 2. Load a Small LLM and Tokenizer

We'll use **Qwen3-0.6B** (~0.6B parameters), which is still under 1B parameters and provides a stronger base model for this sentiment analysis task.

In [ ]:
# Model and tokenizer
model_name = "Qwen/Qwen3-0.6B"

print(f"Loading model: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Qwen models may not define a pad token; fall back to EOS
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
model.to(device)

print("Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 3. Zero-Shot Inference (Baseline)

Before fine-tuning, let's establish a baseline by performing zero-shot inference. We'll use a simple prompt-based approach to classify sentiments.

In [ ]:
def parse_sentiment_label(text):
    """Parse generated answer into 0/1 sentiment label."""
    text = text.strip().lower()
    if text.startswith("positive"):
        return 1
    if text.startswith("negative"):
        return 0
    if "positive" in text and "negative" not in text:
        return 1
    if "negative" in text and "positive" not in text:
        return 0
    return 0


def compute_binary_metrics(y_true, y_pred):
    """Compute accuracy, precision, recall, and F1 for binary classification."""
    tp = sum((yt == 1 and yp == 1) for yt, yp in zip(y_true, y_pred))
    tn = sum((yt == 0 and yp == 0) for yt, yp in zip(y_true, y_pred))
    fp = sum((yt == 0 and yp == 1) for yt, yp in zip(y_true, y_pred))
    fn = sum((yt == 1 and yp == 0) for yt, yp in zip(y_true, y_pred))

    total = len(y_true)
    accuracy = (tp + tn) / total if total > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def zero_shot_inference(texts, labels, num_samples=500):
    """Perform zero-shot sentiment classification on a sample of the test set."""
    model.eval()
    model.to(device)

    # Sample subset for speed
    indices = np.random.choice(len(texts), min(num_samples, len(texts)), replace=False)
    sampled_texts = [texts[i] for i in indices]
    sampled_labels = [labels[i] for i in indices]

    predictions = []

    with torch.no_grad():
        for text in tqdm(sampled_texts, total=len(sampled_texts), desc="Zero-shot inference"):
            text = text[:600]
            prompt = (
                "Classify the sentiment of the movie review as Positive or Negative. "
                "Answer with one word only: Positive or Negative.\n\n"
                f"Review: {text}\n"
                "Sentiment:"
            )

            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768).to(device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )

            # Score only newly generated tokens, not the full prompt text
            new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
            generated_answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
            prediction = parse_sentiment_label(generated_answer)
            predictions.append(prediction)

    return compute_binary_metrics(sampled_labels, predictions)


# Get zero-shot baseline metrics
print("Computing zero-shot baseline...")
baseline_metrics = zero_shot_inference(
    dataset["test"]["text"],
    dataset["test"]["label"],
    num_samples=500
)

print(f"\nZero-shot Accuracy:  {baseline_metrics['accuracy']:.2%}")
print(f"Zero-shot Precision: {baseline_metrics['precision']:.2%}")
print(f"Zero-shot Recall:    {baseline_metrics['recall']:.2%}")
print(f"Zero-shot F1:        {baseline_metrics['f1']:.2%}")

## 4. Prepare Data for Fine-tuning

We'll create a small training set for demonstration purposes and format it for causal language modeling.

In [ ]:
# Use a subset of training data for faster fine-tuning
train_size = 5000
train_dataset = dataset["train"].shuffle(seed=42).select(range(min(train_size, len(dataset["train"]))))

# Format data with sentiment labels
def format_text(examples):
    """
    Format training examples with explicit sentiment labels.
    This helps the model learn to map reviews to sentiments.
    """
    sentiment = ["negative" if label == 0 else "positive" for label in examples["label"]]
    
    texts = [
        f"Review: {text[:300]}\nSentiment: {sent}\n\n"
        for text, sent in zip(examples["text"], sentiment)
    ]
    return {"text": texts}

formatted_dataset = train_dataset.map(format_text, batched=True, remove_columns=train_dataset.column_names)

print(f"Training set size: {len(formatted_dataset)}")
print(f"\nExample formatted text:")
print(formatted_dataset[0]["text"][:300])

## 5. Tokenize Dataset for Training

In [ ]:
def tokenize_function(examples):
    """
    Tokenize text with a fixed max length.
    """
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset.column_names
)

print(f"✓ Tokenization complete!")
print(f"Sample token IDs length: {len(tokenized_dataset[0]['input_ids'])}")

## 6. Configure LoRA

LoRA works by adding low-rank trainable matrices to the model. Here we configure the LoRA parameters:
- **r (rank)**: Dimension of low-rank updates (lower = fewer parameters)
- **lora_alpha**: Scaling parameter
- **target_modules**: Which layers to apply LoRA to
- **lora_dropout**: Dropout for regularization

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=4,  # Rank of low-rank decomposition
    lora_alpha=8,  # Scaling parameter
    target_modules=["q_proj", "v_proj"],  # Common LoRA targets for Qwen attention
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Dropout: {lora_config.lora_dropout}")

## 7. Create LoRA Model

In [ ]:
# Reload the base model fresh for fine-tuning
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)

# Apply LoRA
peft_model = get_peft_model(model, lora_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

# Compare
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

print(f"\nParameter efficiency:")
print(f"  Total: {total_params / 1e9:.2f}B")
print(f"  Trainable: {trainable_params / 1e6:.2f}M")
print(f"  Reduction: {(1 - trainable_params/total_params)*100:.2f}%")

## 8. Fine-tune with LoRA

We'll use a simple training loop to fine-tune the model. In practice, you might use the HuggingFace Trainer, but this shows the core mechanics.

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

# Convert dataset columns to PyTorch tensors so the DataLoader
# can stack them into batched tensors automatically
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Create data loader
batch_size = 16
train_loader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=True)

# Optimizer and learning rate
optimizer = AdamW(peft_model.parameters(), lr=2e-4)

# Training loop
peft_model.to(device)
peft_model.train()

num_epochs = 2
step = 0

print(f"Starting fine-tuning for {num_epochs} epochs...\n")

# Store initial LoRA weight values for verification
initial_lora_weights = {}
for name, param in peft_model.named_parameters():
    if "lora" in name.lower():
        initial_lora_weights[name] = param.detach().clone()

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    epoch_loss = 0

    for batch in tqdm(train_loader, desc="Training"):
        # Move to device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass — use input_ids as labels for causal LM
        outputs = peft_model(**batch, labels=batch["input_ids"])
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.detach().float()
        step += 1

    avg_loss = epoch_loss / len(train_loader)
    print(f"  Average loss: {avg_loss:.4f}\n")

# Verify that LoRA weights were actually updated
weights_changed = False
for name, param in peft_model.named_parameters():
    if "lora" in name.lower() and name in initial_lora_weights:
        weight_diff = (param.detach() - initial_lora_weights[name]).abs().mean().item()
        if weight_diff > 1e-6:
            weights_changed = True
            break

if weights_changed:
    print("✓ LoRA weights were successfully updated during training")
else:
    print("⚠ WARNING: LoRA weights may not have been updated. Check training setup.")

print("Fine-tuning complete!")


## 9. Evaluate Fine-tuned Model

Now let's measure how the fine-tuned model performs on sentiment classification.

In [ ]:
def evaluate_finetuned_model(texts, labels, num_samples=500):
    """Evaluate fine-tuned model on sentiment classification."""
    # Ensure we're using the trained peft_model with LoRA weights
    peft_model.eval()
    peft_model.to(device)
    
    # Verify LoRA is active by checking trainable parameters
    num_lora_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    if num_lora_params == 0:
        print("WARNING: No trainable parameters detected in peft_model. LoRA may not be attached.")
    else:
        print(f"LoRA adapter confirmed: {num_lora_params:,} trainable parameters found")

    # Sample subset
    indices = np.random.choice(len(texts), min(num_samples, len(texts)), replace=False)
    sampled_texts = [texts[i] for i in indices]
    sampled_labels = [labels[i] for i in indices]

    predictions = []

    with torch.no_grad():
        for text in tqdm(sampled_texts, total=len(sampled_texts), desc="Evaluating"):
            text = text[:600]
            prompt = (
                "Classify the sentiment of the movie review as Positive or Negative. "
                "Answer with one word only: Positive or Negative.\n\n"
                f"Review: {text}\n"
                "Sentiment:"
            )

            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768).to(device)
            
            # Ensure inputs and model are on same dtype
            if peft_model.dtype != torch.float32:
                inputs = {k: (v.to(peft_model.dtype) if v.dtype in [torch.float32, torch.float64] else v) 
                         for k, v in inputs.items()}
            
            outputs = peft_model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )

            new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
            generated_answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
            prediction = parse_sentiment_label(generated_answer)
            predictions.append(prediction)

    return compute_binary_metrics(sampled_labels, predictions)


# Evaluate fine-tuned model
print("Evaluating fine-tuned model...")
print("(Using trained LoRA adapter attached to base model)\n")
finetuned_metrics = evaluate_finetuned_model(
    dataset["test"]["text"],
    dataset["test"]["label"],
    num_samples=500
)

print(f"\nFine-tuned Accuracy:  {finetuned_metrics['accuracy']:.2%}")
print(f"Fine-tuned Precision: {finetuned_metrics['precision']:.2%}")
print(f"Fine-tuned Recall:    {finetuned_metrics['recall']:.2%}")
print(f"Fine-tuned F1:        {finetuned_metrics['f1']:.2%}")

## 10. Compare Results

In [ ]:
# Compare baseline vs fine-tuned across multiple metrics
print("=" * 60)
print("RESULTS COMPARISON")
print("=" * 60)

print("Zero-shot Baseline:")
print(f"  Accuracy : {baseline_metrics['accuracy']:.2%}")
print(f"  Precision: {baseline_metrics['precision']:.2%}")
print(f"  Recall   : {baseline_metrics['recall']:.2%}")
print(f"  F1       : {baseline_metrics['f1']:.2%}")

print("\nAfter LoRA Fine-tuning:")
print(f"  Accuracy : {finetuned_metrics['accuracy']:.2%}")
print(f"  Precision: {finetuned_metrics['precision']:.2%}")
print(f"  Recall   : {finetuned_metrics['recall']:.2%}")
print(f"  F1       : {finetuned_metrics['f1']:.2%}")

print("\nAbsolute Improvement (Fine-tuned - Zero-shot):")
print(f"  Accuracy : {(finetuned_metrics['accuracy'] - baseline_metrics['accuracy']):.2%}")
print(f"  Precision: {(finetuned_metrics['precision'] - baseline_metrics['precision']):.2%}")
print(f"  Recall   : {(finetuned_metrics['recall'] - baseline_metrics['recall']):.2%}")
print(f"  F1       : {(finetuned_metrics['f1'] - baseline_metrics['f1']):.2%}")

print("=" * 60)
print("\nKey Insights:")
print("  - LoRA adapts the model for the sentiment analysis task with far fewer trainable parameters.")
print(f"  - Trainable parameters: {trainable_params / 1e6:.2f}M out of {total_params / 1e9:.2f}B total.")
print("  - Use F1 together with accuracy when classes are imbalanced or error types matter.")

## 11. Test on Sample Reviews

In [ ]:
# Test on a few sample reviews
sample_reviews = [
    "This movie was absolutely terrible. Worst film I've ever seen.",
    "Amazing! What a fantastic movie. Loved every minute of it.",
    "It was okay, nothing special but watchable."
]

peft_model.eval()

print("Testing fine-tuned model on sample reviews:\n")

with torch.no_grad():
    for review in sample_reviews:
        prompt = f"Review: {review}\nSentiment:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        sentiment_part = generated.split("Sentiment:")[1].strip()
        
        print(f"Review: {review}")
        print(f"Predicted: {sentiment_part}\n")

## Summary

**What we accomplished:**

1. **Baseline**: Established zero-shot performance on IMDB sentiment classification
2. **LoRA Setup**: Configured low-rank adaptation to train only a fraction of parameters
3. **Fine-tuning**: Trained the LoRA-adapted model for 3 epochs
4. **Evaluation**: Measured improved accuracy on test set
5. **Comparison**: Demonstrated the effectiveness of LoRA fine-tuning

**Why LoRA is important:**

- **Parameter Efficiency**: Train only 0.1-0.5% of parameters instead of 100%
- **Memory Efficiency**: Reduces memory requirements for training
- **Practical**: Makes fine-tuning large models feasible on consumer hardware
- **Quality**: Achieves comparable or better results than full fine-tuning

**Further exploration:**
- Try different LoRA ranks (r) & alpha and see how accuracy changes
- Experiment with different datasets and tasks